In [ ]:
using Random, Statistics, Distributions
using Convex, MosekTools
using LinearAlgebra

# ============================================================
# 1) binning in U with merged tails
# ============================================================

"""
Make edges for U with merged tails:
- total bins = nbins_total
- left tail mass ~ q_bounds[1]
- right tail mass ~ 1-q_bounds[2]
- interior bins are quantile-equal between q_bounds[1] and q_bounds[2]
"""
function make_U_edges_merged_tails(U::Vector{Float64};
    nbins_total::Int = 60,
    q_bounds::Tuple{Float64,Float64} = (0.01, 0.99),
    jitter::Float64 = 0.0,
    rng::AbstractRNG = Random.default_rng(),
)
    @assert nbins_total >= 3 "need at least 3 bins (left tail, interior, right tail)"
    qlo, qhi = q_bounds
    @assert 0 < qlo < qhi < 1

    Uuse = jitter > 0 ? (U .+ jitter .* randn(rng, length(U))) : U

    interior_bins = nbins_total - 2
    probs = collect(range(qlo, qhi; length=interior_bins+1))   # includes qlo, qhi
    qs = quantile(Uuse, probs)                                 # length interior_bins+1
    cuts = qs[2:end-1]                                         # exclude endpoints

    # enforce strictly increasing cuts (handle ties)
    cuts = unique(cuts)
    if length(cuts) < interior_bins-1
        @warn "Many tied quantiles: interior bins reduced from $interior_bins to $(length(cuts)+1)"
    end

    edges = vcat(-Inf, cuts, Inf)
    return edges
end

"""
Assign each U[i] to a bin in 1:nbins (edges length = nbins+1).
"""
function bin_ids(U::Vector{Float64}, edges::Vector{Float64})
    nbins = length(edges) - 1
    b = searchsortedlast.(Ref(edges), U)
    return clamp.(b, 1, nbins)
end

function counts_by_edges(U::Vector{Float64}, edges::Vector{Float64})
    nbins = length(edges)-1
    b = searchsortedlast.(Ref(edges), U)
    b = clamp.(b, 1, nbins)
    counts = zeros(Int, nbins)
    @inbounds for bi in b
        counts[bi] += 1
    end
    return counts
end

# ============================================================
# 2) beta grid for mixture components
# ============================================================

"""
Default grid for mixture on beta.
"""
function default_grid_for_beta(β::Vector{Float64}; n_mu=25, n_sigma=40)
    qlo, qhi = quantile(β, (0.001, 0.999))
    pad = 0.2 * (qhi - qlo)
    mus = collect(range(qlo - pad, qhi + pad; length=n_mu))

    sβ = std(β)
    σ_lo = max(1e-3, 0.05 * sβ)
    σ_hi = max(σ_lo * 2, 2.0 * sβ)
    sigmas = exp.(range(log(σ_lo), log(σ_hi); length=n_sigma))

    return mus, sigmas
end

# ============================================================
# 6) h_hat: compute P(|β| >= t | U-bin) using fitted mixture
# ============================================================

@inline function two_sided_tail(d::UnivariateDistribution, t::Float64)
    tt = abs(t)
    return ccdf(d, tt) + cdf(d, -tt)
end

"""
Given fitted per-bin conditional mixture, compute h(t,v)
where v is D (not log D).
"""
function h_hat(fit, t::Float64, v::Float64)
    u = log(v + 1e-12)
    edges = fit.edges
    nbins = length(edges) - 1
    b = clamp(searchsortedlast(edges, u), 1, nbins)

    π = fit.π_bins[b]
    isempty(π) && return NaN

    meta = fit.comps
    s = 0.0
    @inbounds for k in 1:length(π)
        dk = component_dist(meta, k)
        s += π[k] * two_sided_tail(dk, t)
    end
    return s
end

In [ ]:

# ============================================================
# 3) mixed grid: Normal + Student-t(ν) components
# ============================================================


function build_mixed_grid_separate(
    musN::Vector{Float64}, sigmasN::Vector{Float64};
    musT::Vector{Float64}=musN, sigmasT::Vector{Float64}=sigmasN,
    dofs::Vector{Int}=Int[3,5,8],
    include_normals::Bool=true,
    include_t::Bool=true
)
    mu_list   = Float64[]
    sig_list  = Float64[]
    fam_list  = Symbol[]
    dof_list  = Int[]

    if include_normals
        for μ in musN, σ in sigmasN
            push!(mu_list, μ); push!(sig_list, σ)
            push!(fam_list, :normal); push!(dof_list, 0)
        end
    end

    if include_t
        for ν in dofs, μ in musT, σ in sigmasT
            push!(mu_list, μ); push!(sig_list, σ)
            push!(fam_list, :t); push!(dof_list, ν)
        end
    end

    return (mu=mu_list, sigma=sig_list, family=fam_list, dof=dof_list)
end

@inline function component_dist(meta, k::Int)
    μ = meta.mu[k]
    σ = meta.sigma[k]
    if meta.family[k] === :normal
        return Normal(μ, σ)
    else
        ν = meta.dof[k]
        return LocationScale(μ, σ, TDist(ν))
    end
end

# ============================================================
# 4) fit mixture weights with MOSEK/Convex on fixed grid
# ============================================================

# """
# Fit weights π on a fixed grid of components (Normal + t).
# Objective: maximize Σ_i w_i log( Σ_k π_k f_k(x_i) )
# using log-sum-exp stabilization.

function fit_fixedgrid_mixed_mixture_mosek_weighted(
    x::Vector{Float64},
    w::Vector{Float64};
    musN::Vector{Float64},
    sigmasN::Vector{Float64},
    musT::Vector{Float64}=musN,
    sigmasT::Vector{Float64}=sigmasN,
    dofs::Vector{Int} = Int[3,5,8],
    include_normals::Bool = true,
    include_t::Bool = true,
    verbose::Bool=false
)
    @assert length(x) == length(w)
    N = length(x)

    meta = build_mixed_grid_separate(
        musN, sigmasN;
        musT=musT, sigmasT=sigmasT,
        dofs=dofs,
        include_normals=include_normals,
        include_t=include_t
    )
    K = length(meta.mu)

    logφ = Matrix{Float64}(undef, N, K)
    @inbounds for k in 1:K
        dk = component_dist(meta, k)
        for i in 1:N
            logφ[i, k] = logpdf(dk, x[i])
        end
    end

    m = vec(maximum(logφ, dims=2))
    A = exp.(clamp.(logφ .- m, -745.0, 0.0))

    π = Variable(K)
    constraints = [π >= 0, sum(π) == 1]
    obj = dot(w, m) + sum(w .* log(A * π))

    problem = maximize(obj, constraints)
    solve!(problem, Mosek.Optimizer; silent=!verbose)

    π_hat = vec(evaluate(π))
    π_hat = max.(π_hat, 0.0)
    π_hat ./= sum(π_hat)

    return π_hat, meta
end

In [ ]:
# ============================================================
# 2b) within-bin β-based keep probabilities / weights
# ============================================================

"""
Piecewise keep probability based on z = abs.(βb) within a bin.

- z >= q_beta_tail quantile  -> keep prob = 1
- z <  q_beta_tail quantile  -> keep prob = a_min_beta

Returns:
  aβ::Vector{Float64}, info::NamedTuple
"""
function beta_keep_prob_piecewise(
    βb::Vector{Float64};
    q_beta_tail::Float64 = 0.95,
    a_min_beta::Float64 = 0.10
)
    @assert 0 < q_beta_tail < 1
    @assert 0 < a_min_beta <= 1

    z = abs.(βb)
    zR = quantile(z, q_beta_tail)

    aβ = similar(z, Float64)
    @inbounds for i in eachindex(z)
        aβ[i] = (z[i] >= zR) ? 1.0 : a_min_beta
    end

    info = (
        rule = :piecewise,
        q_beta_tail = q_beta_tail,
        zR = zR,
        a_min_beta = a_min_beta,
        frac_tail = mean(z .>= zR),
        mean_keep_prob = mean(aβ)
    )
    return aβ, info
end


"""
Apply β-based weighting or subsampling inside a bin.

Modes:
- beta_keep_rule = :none       -> returns original βb and all-ones weights
- beta_keep_rule = :piecewise  -> uses beta_keep_prob_piecewise

If beta_use_subsample=true:
  randomly keep points with prob aβ and assign IPW weights = 1/aβ on kept points
Else:
  keep all points and assign weights proportional to 1/aβ (reweight-only)

Returns:
  β_use, w_use, info
"""
function prepare_within_bin_beta_weighting(
    βb::Vector{Float64};
    rng::AbstractRNG = Random.default_rng(),
    beta_keep_rule::Symbol = :none,
    q_beta_tail::Float64 = 0.95,
    a_min_beta::Float64 = 0.10,
    beta_use_subsample::Bool = false,
    rescale_beta_weights::Bool = true
)
    n = length(βb)
    n == 0 && return βb, Float64[], (rule=:none, n_in=0, n_used=0)

    # no weighting
    if beta_keep_rule == :none
        return βb, ones(Float64, n), (rule=:none, n_in=n, n_used=n, mean_weight=1.0, max_weight=1.0)
    end

    # get keep probs
    if beta_keep_rule == :piecewise
        aβ, info0 = beta_keep_prob_piecewise(βb; q_beta_tail=q_beta_tail, a_min_beta=a_min_beta)
    else
        error("Unknown beta_keep_rule=$beta_keep_rule. Supported: :none, :piecewise")
    end

    if beta_use_subsample
        keep = rand(rng, n) .< aβ
        idx = findall(keep)

        β_use = βb[idx]
        w_use = 1.0 ./ aβ[idx]

        if rescale_beta_weights && !isempty(w_use)
            # normalize so sum(weights) ≈ number used
            w_use .*= (length(w_use) / sum(w_use))
        end

        info = (; info0..., n_in=n, n_used=length(β_use),
                 used_frac=length(β_use)/n,
                 mean_weight=(isempty(w_use) ? NaN : mean(w_use)),
                 max_weight=(isempty(w_use) ? NaN : maximum(w_use)))
        return β_use, w_use, info
    else
        # weight-only, no subsampling
        β_use = βb
        w_use = 1.0 ./ aβ

        if rescale_beta_weights && !isempty(w_use)
            w_use .*= (length(w_use) / sum(w_use))
        end

        info = (; info0..., n_in=n, n_used=n, used_frac=1.0,
                 mean_weight=mean(w_use), max_weight=maximum(w_use))
        return β_use, w_use, info
    end
end

In [ ]:
function fit_conditional_beta_given_U_bins(
    β::Vector{Float64},
    U::Vector{Float64};
    nbins_total::Int = 60,
    q_bounds::Tuple{Float64,Float64} = (0.01,0.99),
    jitter::Float64 = 1e-10,
    rng::AbstractRNG = Random.default_rng(),

    # separate grid sizes
    n_mu_N::Int = 25,
    n_sigma_N::Int = 40,
    n_mu_T::Int = 9,
    n_sigma_T::Int = 15,

    dofs::Vector{Int} = Int[3,5,8,12,20],
    include_normals::Bool = true,
    include_t::Bool = true,
    min_bin_size::Int = 500,
    verbose::Bool = false,
    bins_to_fit::Union{Nothing, Vector{Int}} = nothing,

    # optional: widen t sigmas upward for tail focus
    widen_t_sigma_mult::Float64 = 1.0,

    # NEW: within-bin β-based weighting/subsampling
    beta_keep_rule::Symbol = :none,       # :none or :piecewise
    q_beta_tail::Float64 = 0.95,          # top 5% |β| treated as tail
    a_min_beta::Float64 = 0.10,           # center keep prob
    beta_use_subsample::Bool = false,     # false = weight-only; true = actual subsample+IPW
    rescale_beta_weights::Bool = true
)
    @assert length(β) == length(U)

    edges = make_U_edges_merged_tails(U; nbins_total=nbins_total,
                                      q_bounds=q_bounds, jitter=jitter, rng=rng)

    nbins = length(edges) - 1
    b_id = bin_ids(U, edges)

    # shared grids across bins
    musN, sigmasN = default_grid_for_beta(β; n_mu=n_mu_N, n_sigma=n_sigma_N)
    musT, sigmasT = default_grid_for_beta(β; n_mu=n_mu_T, n_sigma=n_sigma_T)

    if widen_t_sigma_mult != 1.0
        sigmasT = exp.(range(log(minimum(sigmasT)),
                             log(maximum(sigmasT) * widen_t_sigma_mult);
                             length=length(sigmasT)))
    end

    π_bins = [Float64[] for _ in 1:nbins]
    counts = zeros(Int, nbins)              # original count in each bin
    counts_used = zeros(Int, nbins)         # actual rows used in fit (after β subsampling if enabled)
    weight_sums = zeros(Float64, nbins)     # sum of weights used in fit
    comps_ref = nothing

    # store diagnostics per bin
    # beta_weight_stats = [nothing for _ in 1:nbins]
    beta_weight_stats = Vector{Any}(undef, nbins)
    fill!(beta_weight_stats, nothing)

    fit_set = bins_to_fit === nothing ? collect(1:nbins) : unique(clamp.(bins_to_fit, 1, nbins))
    fit_flag = falses(nbins)
    fit_flag[fit_set] .= true

    for b in 1:nbins
        idx = findall(==(b), b_id)
        counts[b] = length(idx)

        if !fit_flag[b] || counts[b] < min_bin_size
            continue
        end

        βb = β[idx]

        # NEW: within-bin β-based weighting / subsampling
        β_use, wb, infoβ = prepare_within_bin_beta_weighting(
            βb;
            rng=rng,
            beta_keep_rule=beta_keep_rule,
            q_beta_tail=q_beta_tail,
            a_min_beta=a_min_beta,
            beta_use_subsample=beta_use_subsample,
            rescale_beta_weights=rescale_beta_weights
        )

        beta_weight_stats[b] = infoβ
        counts_used[b] = length(β_use)
        weight_sums[b] = isempty(wb) ? 0.0 : sum(wb)

        # if after subsampling too few points remain, skip
        if counts_used[b] < min_bin_size
            continue
        end

        π_hat, meta = fit_fixedgrid_mixed_mixture_mosek_weighted(
            β_use, wb;
            musN=musN, sigmasN=sigmasN,
            musT=musT, sigmasT=sigmasT,
            dofs=dofs,
            include_normals=include_normals,
            include_t=include_t,
            verbose=verbose
        )

        π_bins[b] = π_hat
        comps_ref === nothing && (comps_ref = meta)
    end

    return (
        edges=edges, π_bins=π_bins, comps=comps_ref,
        counts=counts, counts_used=counts_used, weight_sums=weight_sums,
        bins_fit=fit_set,
        grids=(musN=musN, sigmasN=sigmasN, musT=musT, sigmasT=sigmasT),
        beta_weight_rule=beta_keep_rule,
        beta_use_subsample=beta_use_subsample,
        beta_weight_stats=beta_weight_stats
    )
end

In [ ]:
using Random, Statistics, Distributions

"""
Sigma=1 toy:
    S0 ~ LogNormal(0, a^2)
    beta0 | S0=s0 ~ Normal(0, variance=s0)

Returns:
    beta0, S0, U0=log(S0)
This is the object to use for learning
    P(|beta0| >= t | U0).
"""
function simulate_beta0_S0_toy(B::Int; a::Float64=0.5, rng=Random.default_rng())
    S0 = rand(rng, LogNormal(0.0, a), B)
    beta0 = [rand(rng, Normal(0.0, sqrt(s0))) for s0 in S0]
    U0 = log.(S0 .+ 1e-300)
    return beta0, S0, U0
end

rng = MersenneTwister(123)

In [ ]:
using Random, Statistics, Distributions, Plots

# =========================================================
# 0. Assumes these already exist in your session:
#    - simulate_beta0_S0_toy
#    - make_U_edges_merged_tails
#    - bin_ids
#    - default_grid_for_beta
#    - prepare_within_bin_beta_weighting
#    - fit_fixedgrid_mixed_mixture_mosek_weighted
#    - component_dist
#    - two_sided_tail
# =========================================================

# =========================================================
# 1. Fit only one bin's conditional mixture
# =========================================================
function fit_one_bin_mixture(
    βb::Vector{Float64};
    rng::AbstractRNG = Random.default_rng(),
    n_mu_N::Int = 25,
    n_sigma_N::Int = 30,
    n_mu_T::Int = 10,
    n_sigma_T::Int = 12,
    dofs::Vector{Int} = Int[200, 250, 300],
    include_normals::Bool = true,
    include_t::Bool = true,
    widen_t_sigma_mult::Float64 = 1.0,
    beta_keep_rule::Symbol = :piecewise,
    q_beta_tail::Float64 = 0.99,
    a_min_beta::Float64 = 0.01,
    beta_use_subsample::Bool = true,
    rescale_beta_weights::Bool = true,
    verbose::Bool = false
)
    musN, sigmasN = default_grid_for_beta(βb; n_mu=n_mu_N, n_sigma=n_sigma_N)
    musT, sigmasT = default_grid_for_beta(βb; n_mu=n_mu_T, n_sigma=n_sigma_T)

    if widen_t_sigma_mult != 1.0
        sigmasT = exp.(range(log(minimum(sigmasT)),
                             log(maximum(sigmasT) * widen_t_sigma_mult);
                             length=length(sigmasT)))
    end

    β_use, wb, infoβ = prepare_within_bin_beta_weighting(
        βb;
        rng=rng,
        beta_keep_rule=beta_keep_rule,
        q_beta_tail=q_beta_tail,
        a_min_beta=a_min_beta,
        beta_use_subsample=beta_use_subsample,
        rescale_beta_weights=rescale_beta_weights
    )

    π_hat, meta = fit_fixedgrid_mixed_mixture_mosek_weighted(
        β_use, wb;
        musN=musN, sigmasN=sigmasN,
        musT=musT, sigmasT=sigmasT,
        dofs=dofs,
        include_normals=include_normals,
        include_t=include_t,
        verbose=verbose
    )

    return (π=π_hat, comps=meta, info=infoβ)
end

# learned conditional tail for one fitted bin
function learned_tail_onebin(fit_bin, t::Float64)
    π = fit_bin.π
    meta = fit_bin.comps
    s = 0.0
    @inbounds for k in eachindex(π)
        s += π[k] * two_sided_tail(component_dist(meta, k), t)
    end
    return s
end